# Lesson 6.4 — Unit 6 Recap and Installment C Milestone

Detect-and-localise end to end: every hard fault caught at its seam with a what/where/who, and a healthy run clean — recovery excluded (Unit 7).

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, run_pipeline, localize, failure_event,
    FAILURE_TAXONOMY, REACH_MAX)
W = GreenhouseWorld.demo_row(n=6, seed=1)
ALL_IDS = [f.fid for f in W.fruit]
TGT = understand(model_perception(W, rng=np.random.default_rng(7)), W)[1]
def big(t, j): return 60.0 if (j == 0 and t > 0.3) else 0.0
checks = []
faults = [
    ('occlude',     dict(perception=dict(occlude=ALL_IDS)),   'Understand', 'NO_TARGET'),
    ('block goal',  dict(obstacle=(TGT['xy'], 0.25)),          'Plan',       'PLAN_INVALID'),
    ('disturbance', dict(disturbance=big),                     'Track',      'TRACKING_FAILURE'),
]
for name, kw, stage, code in faults:
    r = run_pipeline(W, W.q.copy(), **kw)
    ev = next(e for e in r['events'] if e['code']==code)
    print('%-12s -> reached=%-10s %s | owner: %s' % (name, r['reached'], code, ev['who']))
    checks.append(r['reached']==stage and any(e['code']==code for e in r['events']))


occlude      -> reached=Understand NO_TARGET | owner: Perceive / Understand (re-perceive or widen search)


block goal   -> reached=Plan       PLAN_INVALID | owner: Plan (replan / relax limits) or Recover


disturbance  -> reached=Track      TRACKING_FAILURE | owner: Execute / Recover


In [2]:
# healthy run reaches Track with no events
r0 = run_pipeline(W, W.q.copy())
print('healthy      -> reached=%s success=%s events=%s' % (r0['reached'], r0['success'], [e['code'] for e in r0['events']]))
checks.append(r0['reached']=='Track' and r0['success'] and r0['events']==[])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('INSTALLMENT C MILESTONE: detect + localise every catalogued fault; recovery is Unit 7.')
print('All checks passed.')


healthy      -> reached=Track success=True events=[]
4/4 checks passed.
INSTALLMENT C MILESTONE: detect + localise every catalogued fault; recovery is Unit 7.
All checks passed.
